# 02. Text Preprocessing & Feature Engineering
## PRG4206 Individual Assignment: AI-Generated Text Detection System

This notebook demonstrates the preprocessing and feature engineering pipelines designed for AI-generated text detection.

### Core Principles:
1. **Punctuation Preservation**: Punctuation is a vital stylistic signal for AI-generated text; we avoid aggressive stripping.
2. **Dual Representation**:
   - Cleaned representation for TF-IDF n-grams (Unicode normalized, HTML/URL normalized).
   - Raw representation for Stylometric feature extraction (preserves exact case, punctuation ratios, paragraph breaks).
3. **No Data Leakage**: TF-IDF vocabulary, IDF weights, and feature scalers are fitted exclusively on the training partition.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.preprocessing import clean_text, preprocess_dataframe
from src.features import extract_stylometric_features, build_tfidf_vectorizer, get_stylometric_feature_names
from src.data_loader import load_splits

print("Modules imported successfully.")


### 1. Test Text Normalization
Demonstrating Unicode normalization, HTML stripping, and whitespace handling.


In [ ]:
sample_raw = "  Hello <b>World</b>! Check https://huggingface.co  with  extra   spaces and zero-width\u200b char. "
sample_clean = clean_text(sample_raw)
print("Original   :", repr(sample_raw))
print("Normalized :", repr(sample_clean))


### 2. Stylometric Feature Extraction
Extracting 21 fine-grained surface and lexical features on human vs AI samples.


In [ ]:
sample_ai = ("The concept of machine learning involves algorithms that improve automatically through "
             "experience. Specifically, deep learning architectures such as convolutional and recurrent neural "
             "networks have advanced computer vision and natural language processing tasks.")

sample_human = ("so basically ML is when you give a computer lots of examples and it figures out the rules itself lol. "
                "like, instead of programming every detail you just show it 1000 pictures of dogs and cats!")

ai_feats = extract_stylometric_features(sample_ai)
human_feats = extract_stylometric_features(sample_human)

feat_comparison = pd.DataFrame({"AI Text": ai_feats, "Human Text": human_feats})
feat_comparison.round(4)


### 3. Load Verified Splits & Verify Leakage Prevention


In [ ]:
train_df, val_df, test_df = load_splits()
print(f"Train samples : {len(train_df):,}")
print(f"Val samples   : {len(val_df):,}")
print(f"Test samples  : {len(test_df):,}")

# Verify no overlap
overlap = set(train_df['text']).intersection(set(test_df['text']))
print(f"Train/Test text overlap: {len(overlap)} (Must be 0)")
assert len(overlap) == 0, "Data leakage detected!"
